# Feature Engineering

This notebook creates feature-engineered train/test datasets. Feature rules are defined without using the target as an input. `health_condition` is used only for train-side analysis to evaluate whether candidate features separate the target classes.

In [1]:
import re

import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

## Load Latest Prepared Data

Default input is `train_preprocessed.csv` / `test_preprocessed.csv`, which should already have null imputation and preprocessing applied. If those files are unavailable, fall back to imputed files, then raw files.

In [2]:
ID_COL = "id"
TARGET_COL = "health_condition"

DATA_CANDIDATES = [
    ("data/train_preprocessed.csv", "data/test_preprocessed.csv", "preprocessed"),
    ("data/train_imputed.csv", "data/test_imputed.csv", "imputed"),
    ("data/train.csv", "data/test.csv", "raw"),
]

for train_path, test_path, data_stage in DATA_CANDIDATES:
    if pd.io.common.file_exists(train_path) and pd.io.common.file_exists(test_path):
        break
else:
    raise FileNotFoundError("Could not find matching train/test CSV files.")

train_base = pd.read_csv(train_path)
test_base = pd.read_csv(test_path)

print("data stage:", data_stage)
print("train path:", train_path)
print("test path:", test_path)
print("train shape:", train_base.shape)
print("test shape:", test_base.shape)

data stage: preprocessed
train path: data/train_preprocessed.csv
test path: data/test_preprocessed.csv
train shape: (690088, 23)
test shape: (295753, 22)


In [3]:
feature_cols = [col for col in train_base.columns if col not in [ID_COL, TARGET_COL]]
missing_in_test = sorted(set(feature_cols) - set(test_base.columns))
extra_in_test = sorted(set(test_base.columns) - set(feature_cols) - {ID_COL})

print("feature columns:", len(feature_cols))
print("missing in test:", missing_in_test)
print("extra in test:", extra_in_test)

feature columns: 21
missing in test: []
extra in test: []


## Feature Engineering Helpers

In [4]:
def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    result = numerator / denominator.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan).fillna(0)


def add_bmi_category(df):
    bins = [-np.inf, 18.5, 25.0, 30.0, np.inf]
    labels = ["underweight", "normal", "overweight", "obese"]
    return pd.cut(df["bmi"], bins=bins, labels=labels, right=False).astype("object")


def add_engineered_features(df):
    df = df.copy()

    stress_map = {"low": 0, "medium": 1, "high": 2}
    sleep_quality_map = {"poor": 0, "average": 1, "good": 2}
    activity_map = {"sedentary": 0, "moderate": 1, "active": 2}
    smoking_alcohol_map = {"no": 0, "occasional": 1, "yes": 2}

    df["stress_score"] = df["stress_level"].map(stress_map).fillna(-1).astype(int)
    df["sleep_quality_score"] = df["sleep_quality"].map(sleep_quality_map).fillna(-1).astype(int)
    df["physical_activity_score"] = df["physical_activity_level"].map(activity_map).fillna(-1).astype(int)
    df["smoking_alcohol_score"] = df["smoking_alcohol"].map(smoking_alcohol_map).fillna(-1).astype(int)

    df["sleep_quality_risk"] = (2 - df["sleep_quality_score"]).clip(lower=0)
    df["activity_risk"] = (2 - df["physical_activity_score"]).clip(lower=0)

    df["sleep_deficit_from_8"] = (df["sleep_duration"] - 8).abs()
    df["sleep_outside_7_9"] = np.where(df["sleep_duration"] < 7, 7 - df["sleep_duration"], 0) + np.where(df["sleep_duration"] > 9, df["sleep_duration"] - 9, 0)
    df["is_low_sleep"] = (df["sleep_duration"] < 6).astype(int)
    df["is_high_sleep"] = (df["sleep_duration"] > 9).astype(int)

    df["bmi_category"] = add_bmi_category(df)
    df["bmi_risk_score"] = np.select(
        [df["bmi"] < 18.5, df["bmi"] < 25, df["bmi"] < 30, df["bmi"] >= 30],
        [1, 0, 1, 2],
        default=0,
    )
    df["is_high_bmi"] = (df["bmi"] >= 25).astype(int)
    df["is_obese_bmi"] = (df["bmi"] >= 30).astype(int)

    df["heart_rate_distance_from_75"] = (df["heart_rate"] - 75).abs()
    df["is_low_heart_rate"] = (df["heart_rate"] < 60).astype(int)
    df["is_high_heart_rate"] = (df["heart_rate"] > 90).astype(int)

    df["is_low_water"] = (df["water_intake"] < 1.5).astype(int)
    df["is_high_water"] = (df["water_intake"] > 3.0).astype(int)

    df["steps_per_exercise_min"] = safe_divide(df["step_count"], df["exercise_duration"] + 1)
    df["exercise_min_per_1000_steps"] = safe_divide(df["exercise_duration"], df["step_count"] / 1000 + 1)
    df["calories_per_step"] = safe_divide(df["calorie_expenditure"], df["step_count"] + 1)
    df["calories_per_exercise_min"] = safe_divide(df["calorie_expenditure"], df["exercise_duration"] + 1)
    df["water_per_1000_calories"] = safe_divide(df["water_intake"], df["calorie_expenditure"] / 1000)
    df["water_per_exercise_min"] = safe_divide(df["water_intake"], df["exercise_duration"] + 1)

    df["diet_is_balanced"] = (df["diet_type"] == "balanced").astype(int)
    df["diet_is_veg"] = (df["diet_type"] == "veg").astype(int)
    df["diet_is_non_veg"] = (df["diet_type"] == "non-veg").astype(int)

    df["poor_sleep_high_stress"] = ((df["sleep_quality_risk"] >= 2) & (df["stress_score"] >= 2)).astype(int)
    df["low_activity_high_bmi"] = ((df["activity_risk"] >= 2) & (df["bmi"] >= 25)).astype(int)
    df["low_sleep_high_stress"] = ((df["is_low_sleep"] == 1) & (df["stress_score"] >= 2)).astype(int)
    df["high_bmi_low_activity"] = ((df["is_high_bmi"] == 1) & (df["activity_risk"] >= 1)).astype(int)

    df["lifestyle_risk_score"] = (
        df["stress_score"].clip(lower=0)
        + df["sleep_quality_risk"]
        + df["activity_risk"]
        + df["smoking_alcohol_score"].clip(lower=0)
        + df["is_low_sleep"]
        + df["is_low_water"]
    )
    df["metabolic_risk_score"] = (
        df["bmi_risk_score"]
        + df["is_high_heart_rate"]
        + df["is_low_heart_rate"]
        + df["is_high_bmi"]
        + df["activity_risk"]
    )
    df["recovery_score"] = (
        df["sleep_quality_score"].clip(lower=0)
        + df["physical_activity_score"].clip(lower=0)
        + (1 - df["is_low_sleep"])
        + (1 - df["is_low_water"])
    )

    return df

## Apply Same Feature Rules To Train And Test

In [5]:
train_fe = add_engineered_features(train_base)
test_fe = add_engineered_features(test_base)

new_feature_cols = [col for col in train_fe.columns if col not in train_base.columns]

print("new feature count:", len(new_feature_cols))
print("new features:")
for col in new_feature_cols:
    print("-", col)

print("train_fe shape:", train_fe.shape)
print("test_fe shape:", test_fe.shape)

new feature count: 35
new features:
- stress_score
- sleep_quality_score
- physical_activity_score
- smoking_alcohol_score
- sleep_quality_risk
- activity_risk
- sleep_deficit_from_8
- sleep_outside_7_9
- is_low_sleep
- is_high_sleep
- bmi_category
- bmi_risk_score
- is_high_bmi
- is_obese_bmi
- heart_rate_distance_from_75
- is_low_heart_rate
- is_high_heart_rate
- is_low_water
- is_high_water
- steps_per_exercise_min
- exercise_min_per_1000_steps
- calories_per_step
- calories_per_exercise_min
- water_per_1000_calories
- water_per_exercise_min
- diet_is_balanced
- diet_is_veg
- diet_is_non_veg
- poor_sleep_high_stress
- low_activity_high_bmi
- low_sleep_high_stress
- high_bmi_low_activity
- lifestyle_risk_score
- metabolic_risk_score
- recovery_score
train_fe shape: (690088, 58)
test_fe shape: (295753, 57)


In [6]:
train_only_features = sorted(set(train_fe.columns) - set(test_fe.columns) - {TARGET_COL})
test_only_features = sorted(set(test_fe.columns) - set(train_fe.columns))

print("train-only feature columns excluding target:", train_only_features)
print("test-only feature columns:", test_only_features)

assert not train_only_features
assert not test_only_features
assert TARGET_COL not in test_fe.columns

train-only feature columns excluding target: []
test-only feature columns: []


## Target-Based Feature Review On Train Only

These tables use `health_condition` only to evaluate candidate features. The target is not used to create train/test feature values.

In [7]:
engineered_numeric_cols = [
    col for col in new_feature_cols
    if pd.api.types.is_numeric_dtype(train_fe[col])
]
engineered_categorical_cols = [
    col for col in new_feature_cols
    if not pd.api.types.is_numeric_dtype(train_fe[col])
]

feature_target_summary = train_fe.groupby(TARGET_COL)[engineered_numeric_cols].agg(["mean", "std"]).T
feature_target_summary

health_condition                      at-risk         fit   unhealthy
stress_score                mean     0.989847    0.170640    1.850062
                            std      0.663850    0.415790    0.378906
sleep_quality_score         mean     1.011071    1.237821    0.609365
                            std      0.777814    0.741818    0.674724
physical_activity_score     mean     0.926436    1.903424    1.008489
                            std      0.775544    0.337540    0.787396
smoking_alcohol_score       mean     1.041113    0.815692    1.268433
                            std      0.826124    0.805768    0.782428
sleep_quality_risk          mean     0.988929    0.762179    1.390635
                            std      0.777814    0.741818    0.674724
activity_risk               mean     1.073564    0.096576    0.991511
                            std      0.775544    0.337540    0.787396
sleep_deficit_from_8        mean     1.181270    0.663505    2.463638
                            std      0.789142    0.460116    0.786448
sleep_outside_7_9           mean     0.386657    0.064220    1.469285
                            std      0.607404    0.231391    0.773497
is_low_sleep                mean     0.143211    0.007839    0.858984
                            std      0.350288    0.088189    0.348041
is_high_sleep               mean     0.050074    0.093938    0.001836
                            std      0.218098    0.291746    0.042813
bmi_risk_score              mean     0.233321    0.206366    0.345125
                            std      0.424105    0.404702    0.524976
is_high_bmi                 mean     0.200904    0.085044    0.320335
                            std      0.400677    0.278950    0.466609
is_obese_bmi                mean     0.000491    0.000000    0.024790
                            std      0.022155    0.000000    0.155487
heart_rate_distance_from_75 mean     6.522392    6.557108    6.335237
                            std      4.880122    4.890082    4.750261
is_low_heart_rate           mean     0.030005    0.026254    0.023716
                            std      0.170602    0.159893    0.152165
is_high_heart_rate          mean     0.034126    0.038565    0.034630
                            std      0.181554    0.192558    0.182843
is_low_water                mean     0.064270    0.067407    0.065293
                            std      0.245234    0.250729    0.247045
is_high_water               mean     0.070919    0.077909    0.071201
                            std      0.256690    0.268031    0.257162
steps_per_exercise_min      mean   384.805342  242.697944  371.775457
                            std   1017.801752  191.798464  978.310861
exercise_min_per_1000_steps mean     4.761979    4.178705    4.724670
                            std      2.973688    1.425967    2.875068
calories_per_step           mean     0.374315    0.218294    0.362947
                            std      0.318941    0.091920    0.308840
calories_per_exercise_min   mean   121.802305   50.601654  115.931145
                            std    331.999346   70.167969  319.190401
water_per_1000_calories     mean     1.011877    0.940926    0.998481
                            std      0.290058    0.255845    0.285722
water_per_exercise_min      mean     0.127525    0.047841    0.120345
                            std      0.367225    0.086410    0.352776
diet_is_balanced            mean     0.325993    0.348818    0.343583
                            std      0.468745    0.476602    0.474908
diet_is_veg                 mean     0.344690    0.359747    0.342388
                            std      0.475268    0.479932    0.474513
diet_is_non_veg             mean     0.329316    0.291435    0.314029
                            std      0.469965    0.454429    0.464132
poor_sleep_high_stress      mean     0.054896    0.005025    0.431917
                            std      0.227777    0.070708    0.495347
low_activi

In [8]:
target_counts_by_bmi_category = pd.crosstab(
    train_fe["bmi_category"],
    train_fe[TARGET_COL],
    normalize="index",
).round(4)

target_counts_by_bmi_category

health_condition,at-risk,fit,unhealthy
bmi_category,,,
normal,0.8652,0.0601,0.0747
obese,0.1690,0.0000,0.8310
overweight,0.8531,0.0243,0.1226
underweight,0.7966,0.2034,0.0000


In [9]:
binary_like_cols = [
    col for col in engineered_numeric_cols
    if set(train_fe[col].dropna().unique()).issubset({0, 1})
]

binary_feature_rates_by_target = train_fe.groupby(TARGET_COL)[binary_like_cols].mean().T.sort_index()
binary_feature_rates_by_target.style.format("{:.3f}")

health_condition,at-risk,fit,unhealthy
diet_is_balanced,0.326,0.349,0.344
diet_is_non_veg,0.329,0.291,0.314
diet_is_veg,0.345,0.360,0.342
high_bmi_low_activity,0.148,0.007,0.219
is_high_bmi,0.201,0.085,0.320
is_high_heart_rate,0.034,0.039,0.035
is_high_sleep,0.050,0.094,0.002
is_high_water,0.071,0.078,0.071
is_low_heart_rate,0.030,0.026,0.024
is_low_sleep,0.143,0.008,0.859


## Mutual Information Check

This is a train-only relevance check. It ranks engineered features by estimated relationship with `health_condition`. It does not add target-derived features.

In [10]:
MI_SAMPLE_SIZE = 200_000
RANDOM_STATE = 42

mi_data = train_fe[[TARGET_COL] + new_feature_cols].copy()
if len(mi_data) > MI_SAMPLE_SIZE:
    mi_data = mi_data.sample(MI_SAMPLE_SIZE, random_state=RANDOM_STATE)

X_mi = mi_data[new_feature_cols].copy()
y_mi = LabelEncoder().fit_transform(mi_data[TARGET_COL])

discrete_features = []
for col in new_feature_cols:
    if not pd.api.types.is_numeric_dtype(X_mi[col]):
        X_mi[col] = pd.factorize(X_mi[col], sort=True)[0]
        discrete_features.append(True)
    else:
        X_mi[col] = X_mi[col].fillna(X_mi[col].median())
        discrete_features.append(X_mi[col].nunique() <= 20)

mi_scores = mutual_info_classif(
    X_mi,
    y_mi,
    discrete_features=discrete_features,
    random_state=RANDOM_STATE,
)

mi_report = (
    pd.DataFrame({"feature": new_feature_cols, "mutual_information": mi_scores})
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

mi_report.style.format({"mutual_information": "{:.5f}"})

,feature,mutual_information
0,low_sleep_high_stress,0.18154
1,sleep_deficit_from_8,0.14228
2,sleep_outside_7_9,0.13735
3,lifestyle_risk_score,0.13731
4,stress_score,0.12254
5,is_low_sleep,0.10356
6,recovery_score,0.06721
7,activity_risk,0.05152
8,physical_activity_score,0.05152
9,poor_sleep_high_stress,0.04614


## Numeric Encoding For Modeling

String categorical columns are useful for analysis, but most sklearn models need numeric inputs. This section learns category levels from train only, creates matching one-hot columns for train/test, and builds numeric-only feature datasets. Unknown test categories become all zeros for that categorical field.

In [11]:
def safe_category_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^0-9a-zA-Z]+", "_", value).strip("_")
    return value or "missing"


categorical_cols_to_encode = [
    col for col in train_fe.columns
    if col != TARGET_COL and not pd.api.types.is_numeric_dtype(train_fe[col])
]

category_levels = {
    col: sorted(train_fe[col].dropna().astype(str).unique().tolist())
    for col in categorical_cols_to_encode
}


def add_one_hot_from_train_levels(df, category_levels):
    df = df.copy()
    encoded_cols = []

    for col, levels in category_levels.items():
        values = df[col].astype(str)
        for level in levels:
            encoded_col = f"{col}__{safe_category_name(level)}"
            df[encoded_col] = (values == level).astype(int)
            encoded_cols.append(encoded_col)

    return df, encoded_cols


train_fe_encoded, encoded_categorical_cols = add_one_hot_from_train_levels(train_fe, category_levels)
test_fe_encoded, _ = add_one_hot_from_train_levels(test_fe, category_levels)

train_features_numeric = train_fe_encoded.drop(columns=categorical_cols_to_encode)
test_features_numeric = test_fe_encoded.drop(columns=[
    col for col in categorical_cols_to_encode
    if col in test_fe_encoded.columns
])

train_non_numeric = train_features_numeric.drop(columns=[TARGET_COL]).select_dtypes(exclude="number").columns.tolist()
test_non_numeric = test_features_numeric.select_dtypes(exclude="number").columns.tolist()

train_only_numeric_cols = sorted(set(train_features_numeric.columns) - set(test_features_numeric.columns) - {TARGET_COL})
test_only_numeric_cols = sorted(set(test_features_numeric.columns) - set(train_features_numeric.columns))

print("categorical columns encoded:", categorical_cols_to_encode)
print("encoded categorical feature count:", len(encoded_categorical_cols))
print("non-numeric train feature columns excluding target:", train_non_numeric)
print("non-numeric test feature columns:", test_non_numeric)
print("train-only numeric columns excluding target:", train_only_numeric_cols)
print("test-only numeric columns:", test_only_numeric_cols)
print("train_features_numeric shape:", train_features_numeric.shape)
print("test_features_numeric shape:", test_features_numeric.shape)

assert not train_non_numeric
assert not test_non_numeric
assert not train_only_numeric_cols
assert not test_only_numeric_cols
assert TARGET_COL not in test_features_numeric.columns

categorical columns encoded: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category']
encoded categorical feature count: 22
non-numeric train feature columns excluding target: []
non-numeric test feature columns: []
train-only numeric columns excluding target: []
test-only numeric columns: []
train_features_numeric shape: (690088, 73)
test_features_numeric shape: (295753, 72)


## Final Save

Save feature-engineered datasets. `health_condition` remains only in train.

In [12]:
train_output_path = "data/train_features.csv"
test_output_path = "data/test_features.csv"
train_numeric_output_path = "data/train_features_numeric.csv"
test_numeric_output_path = "data/test_features_numeric.csv"

train_fe.to_csv(train_output_path, index=False)
test_fe.to_csv(test_output_path, index=False)
train_features_numeric.to_csv(train_numeric_output_path, index=False)
test_features_numeric.to_csv(test_numeric_output_path, index=False)

print("saved:", train_output_path, train_fe.shape)
print("saved:", test_output_path, test_fe.shape)
print("saved:", train_numeric_output_path, train_features_numeric.shape)
print("saved:", test_numeric_output_path, test_features_numeric.shape)

saved: data/train_features.csv (690088, 58)
saved: data/test_features.csv (295753, 57)
saved: data/train_features_numeric.csv (690088, 73)
saved: data/test_features_numeric.csv (295753, 72)


In [13]:
train_fe.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier,stress_score,sleep_quality_score,physical_activity_score,smoking_alcohol_score,sleep_quality_risk,activity_risk,sleep_deficit_from_8,sleep_outside_7_9,is_low_sleep,is_high_sleep,bmi_category,bmi_risk_score,is_high_bmi,is_obese_bmi,heart_rate_distance_from_75,is_low_heart_rate,is_high_heart_rate,is_low_water,is_high_water,steps_per_exercise_min,exercise_min_per_1000_steps,calories_per_step,calories_per_exercise_min,water_per_1000_calories,water_per_exercise_min,diet_is_balanced,diet_is_veg,diet_is_non_veg,poor_sleep_high_stress,low_activity_high_bmi,low_sleep_high_stress,high_bmi_low_activity,lifestyle_risk_score,metabolic_risk_score,recovery_score
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female,0,0,0,0,0,0,0,0,2,1,0,2,1,2,2.78,1.78,1,0,overweight,1,1,0,4.4,0,0,0,0,63.750000,8.512468,1.638282,104.519231,0.855566,0.089423,0,1,0,0,1,1,1,8,4,2
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other,0,0,0,0,0,0,0,0,0,1,1,2,1,1,2.47,1.47,1,0,overweight,1,1,0,3.7,0,0,1,0,194.322200,4.581765,0.198746,38.624754,0.640895,0.024754,0,0,1,0,0,0,1,6,3,2
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male,0,0,0,0,0,0,0,0,2,0,2,2,2,0,2.71,1.71,1,0,normal,0,0,0,0.4,0,0,0,0,363.580563,2.503943,0.189069,68.746803,0.595238,0.040921,0,1,0,1,0,1,0,7,0,3
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female,0,0,0,0,0,0,0,0,2,1,2,1,1,0,3.30,2.30,1,0,normal,0,0,0,2.2,0,0,0,0,117.799672,7.328114,0.366551,43.185550,0.768061,0.033169,0,1,0,0,0,1,0,5,0,4
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,medium,average,sedentary,yes,male,0,0,0,0,0,0,0,0,1,1,0,2,1,2,0.77,0.00,0,0,overweight,1,1,0,1.6,0,0,0,0,140.085106,6.065401,0.388762,54.468085,0.878906,0.047872,0,1,0,0,1,0,1,6,4,3


In [14]:
test_fe.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier,stress_score,sleep_quality_score,physical_activity_score,smoking_alcohol_score,sleep_quality_risk,activity_risk,sleep_deficit_from_8,sleep_outside_7_9,is_low_sleep,is_high_sleep,bmi_category,bmi_risk_score,is_high_bmi,is_obese_bmi,heart_rate_distance_from_75,is_low_heart_rate,is_high_heart_rate,is_low_water,is_high_water,steps_per_exercise_min,exercise_min_per_1000_steps,calories_per_step,calories_per_exercise_min,water_per_1000_calories,water_per_exercise_min,diet_is_balanced,diet_is_veg,diet_is_non_veg,poor_sleep_high_stress,low_activity_high_bmi,low_sleep_high_stress,high_bmi_low_activity,lifestyle_risk_score,metabolic_risk_score,recovery_score
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male,0,0,0,0,0,0,0,0,2,0,2,1,2,0,2.65,1.65,1,0,normal,0,0,0,10.1,0,0,0,0,234.165289,3.922991,0.193746,45.371901,0.677596,0.030744,0,1,0,1,0,1,0,6,0,3
1,690089,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other,0,0,0,0,0,0,0,0,2,0,0,2,2,2,1.01,0.01,0,0,normal,0,0,0,8.1,0,0,0,0,266.705882,3.140623,0.260659,69.529412,1.353638,0.094118,1,0,0,1,0,0,0,8,2,2
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,male,0,0,0,1,0,0,0,1,1,0,2,0,2,0,1.32,0.32,0,0,normal,0,0,0,15.3,1,0,0,0,267.676768,3.403509,0.229417,61.414141,0.907895,0.055758,1,0,0,0,0,0,0,3,1,4
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other,0,0,0,0,0,0,0,0,0,2,1,2,0,1,0.87,0.00,0,0,overweight,1,1,0,3.5,0,0,0,0,109.343696,7.761560,0.393872,43.074266,0.938252,0.040415,0,1,0,0,0,0,1,3,3,5
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other,0,0,0,0,0,0,0,0,2,1,2,1,1,0,2.51,1.51,1,0,normal,0,0,0,2.7,0,0,0,0,343.910891,2.645361,0.131558,45.247525,1.340263,0.060644,0,1,0,0,0,1,0,5,0,4


In [15]:
train_features_numeric.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier,stress_score,sleep_quality_score,physical_activity_score,smoking_alcohol_score,sleep_quality_risk,activity_risk,sleep_deficit_from_8,sleep_outside_7_9,is_low_sleep,is_high_sleep,bmi_risk_score,is_high_bmi,is_obese_bmi,heart_rate_distance_from_75,is_low_heart_rate,is_high_heart_rate,is_low_water,is_high_water,steps_per_exercise_min,exercise_min_per_1000_steps,calories_per_step,calories_per_exercise_min,water_per_1000_calories,water_per_exercise_min,diet_is_balanced,diet_is_veg,diet_is_non_veg,poor_sleep_high_stress,low_activity_high_bmi,low_sleep_high_stress,high_bmi_low_activity,lifestyle_risk_score,metabolic_risk_score,recovery_score,diet_type__balanced,diet_type__non_veg,diet_type__veg,stress_level__high,stress_level__low,stress_level__medium,sleep_quality__average,sleep_quality__good,sleep_quality__poor,physical_activity_level__active,physical_activity_level__moderate,physical_activity_level__sedentary,smoking_alcohol__no,smoking_alcohol__occasional,smoking_alcohol__yes,gender__female,gender__male,gender__other,bmi_category__normal,bmi_category__obese,bmi_category__overweight,bmi_category__underweight
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,0,0,0,0,0,0,0,0,2,1,0,2,1,2,2.78,1.78,1,0,1,1,0,4.4,0,0,0,0,63.750000,8.512468,1.638282,104.519231,0.855566,0.089423,0,1,0,0,1,1,1,8,4,2,0,0,1,1,0,0,1,0,0,0,0,1,0,0,1,1,0,0,0,0,1,0
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,0,0,0,0,0,0,0,0,0,1,1,2,1,1,2.47,1.47,1,0,1,1,0,3.7,0,0,1,0,194.322200,4.581765,0.198746,38.624754,0.640895,0.024754,0,0,1,0,0,0,1,6,3,2,0,1,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,0,0,0,0,0,0,0,0,2,0,2,2,2,0,2.71,1.71,1,0,0,0,0,0.4,0,0,0,0,363.580563,2.503943,0.189069,68.746803,0.595238,0.040921,0,1,0,1,0,1,0,7,0,3,0,0,1,1,0,0,0,0,1,1,0,0,0,0,1,0,1,0,1,0,0,0
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,0,0,0,0,0,0,0,0,2,1,2,1,1,0,3.30,2.30,1,0,0,0,0,2.2,0,0,0,0,117.799672,7.328114,0.366551,43.185550,0.768061,0.033169,0,1,0,0,0,1,0,5,0,4,0,0,1,1,0,0,1,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,0,0,0,0,0,0,0,0,1,1,0,2,1,2,0.77,0.00,0,0,1,1,0,1.6,0,0,0,0,140.085106,6.065401,0.388762,54.468085,0.878906,0.047872,0,1,0,0,1,0,1,6,4,3,0,0,1,0,0,1,1,0,0,0,0,1,0,0,1,0,1,0,0,0,1,0


In [16]:
test_features_numeric.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier,stress_score,sleep_quality_score,physical_activity_score,smoking_alcohol_score,sleep_quality_risk,activity_risk,sleep_deficit_from_8,sleep_outside_7_9,is_low_sleep,is_high_sleep,bmi_risk_score,is_high_bmi,is_obese_bmi,heart_rate_distance_from_75,is_low_heart_rate,is_high_heart_rate,is_low_water,is_high_water,steps_per_exercise_min,exercise_min_per_1000_steps,calories_per_step,calories_per_exercise_min,water_per_1000_calories,water_per_exercise_min,diet_is_balanced,diet_is_veg,diet_is_non_veg,poor_sleep_high_stress,low_activity_high_bmi,low_sleep_high_stress,high_bmi_low_activity,lifestyle_risk_score,metabolic_risk_score,recovery_score,diet_type__balanced,diet_type__non_veg,diet_type__veg,stress_level__high,stress_level__low,stress_level__medium,sleep_quality__average,sleep_quality__good,sleep_quality__poor,physical_activity_level__active,physical_activity_level__moderate,physical_activity_level__sedentary,smoking_alcohol__no,smoking_alcohol__occasional,smoking_alcohol__yes,gender__female,gender__male,gender__other,bmi_category__normal,bmi_category__obese,bmi_category__overweight,bmi_category__underweight
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,0,0,0,0,0,0,0,0,2,0,2,1,2,0,2.65,1.65,1,0,0,0,0,10.1,0,0,0,0,234.165289,3.922991,0.193746,45.371901,0.677596,0.030744,0,1,0,1,0,1,0,6,0,3,0,0,1,1,0,0,0,0,1,1,0,0,0,1,0,0,1,0,1,0,0,0
1,690089,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,0,0,0,0,0,0,0,0,2,0,0,2,2,2,1.01,0.01,0,0,0,0,0,8.1,0,0,0,0,266.705882,3.140623,0.260659,69.529412,1.353638,0.094118,1,0,0,1,0,0,0,8,2,2,1,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,0,0
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,0,0,0,1,0,0,0,1,1,0,2,0,2,0,1.32,0.32,0,0,0,0,0,15.3,1,0,0,0,267.676768,3.403509,0.229417,61.414141,0.907895,0.055758,1,0,0,0,0,0,0,3,1,4,1,0,0,0,0,1,0,0,1,1,0,0,1,0,0,0,1,0,1,0,0,0
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,0,0,0,0,0,0,0,0,0,2,1,2,0,1,0.87,0.00,0,0,1,1,0,3.5,0,0,0,0,109.343696,7.761560,0.393872,43.074266,0.938252,0.040415,0,1,0,0,0,0,1,3,3,5,0,0,1,0,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0,0,1,0
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,0,0,0,0,0,0,0,0,2,1,2,1,1,0,2.51,1.51,1,0,0,0,0,2.7,0,0,0,0,343.910891,2.645361,0.131558,45.247525,1.340263,0.060644,0,1,0,0,0,1,0,5,0,4,0,0,1,1,0,0,1,0,0,1,0,0,0,1,0,0,0,1,1,0,0,0
